# Challenge 6: Quantum Art Studio ★☆☆

**The mission:** Use qubits as your paintbrush. Every pixel's color comes from measuring qubits, so every piece you make is one of a kind. You'll make noise art, a night sky, entangled mirror images, and a gradient. Then you'll create a gallery piece with a title and an artist statement.

**Quantum ideas:** superposition, rotation angles and probability, entanglement.
**Best for:** art, design, and music majors; visual thinkers; first-time coders.
**Time plan:** Missions 1 to 5 take about 60 to 75 minutes. Spend the rest on your gallery piece.

The setup cell gives you `show_art(grid, colors, title)`. Colors can be names like `"black"` or `"gold"`, or hex codes like `"#ff6f91"`. Change any palette you like.

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
from matplotlib.colors import ListedColormap

def show_art(grid, colors, title=""):
    """Show a 2D grid of numbers as a picture. colors[0] paints 0s, colors[1] paints 1s, and so on."""
    fig, ax = plt.subplots(figsize=(6, 6 * len(grid) / len(grid[0])))
    ax.imshow(np.array(grid), cmap=ListedColormap(colors), vmin=0, vmax=len(colors) - 1, interpolation="nearest")
    ax.set_title(title, fontsize=14)
    ax.axis("off")
    plt.show()

def check_mission_1(qc):
    _check_probs(qc, {"0": 0.5, "1": 0.5}, "Mission 1 complete: quantum noise!", "qc.h(0) makes each pixel a quantum coin flip.")

def check_mission_2(qc):
    _check_probs(qc, {"00": .25, "01": .25, "10": .25, "11": .25}, "Mission 2 complete: four quantum colors.",
                 "both qubits need an H gate. qc.h([0, 1]) does both at once.")

def check_mission_3(qc):
    _check_probs(qc, {"0": 0.95, "1": 0.05}, "Mission 3 complete: a sky with 5% stars.",
                 "solve sin(theta/2)^2 = star_chance for theta. math.asin(x) gives the angle whose sin is x.")

def check_mission_4(qc):
    _check_probs(qc, {"00": 0.5, "11": 0.5}, "Mission 4 complete: entangled symmetry!",
                 "entangled.h(0), then entangled.cx(0, 1).")

def check_mission_5(grid):
    g = np.array(grid)
    ok = g[:, 0].sum() == 0 and g[:, -1].sum() == g.shape[0] and 0.3 < g.mean() < 0.7
    _report(ok, "Mission 5 complete: a quantum gradient.",
            "theta should be 0 when col is 0 and math.pi when col is WIDTH - 1. Try math.pi * col / (WIDTH - 1).")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## Mission 1: Quantum noise

Each pixel is one measurement of a qubit: 0 paints black, 1 paints white. `run_many` gives you 256 results at once, and NumPy folds the list into a 16 × 16 square.

**TODO:** Put the qubit in superposition so every pixel is a quantum coin flip.

In [ ]:
SIZE = 16   # the canvas is 16 x 16 pixels

qc = QuantumCircuit(1)
# TODO: put the qubit in superposition

qc.measure_all()

pixels = [int(b) for b in run_many(qc, SIZE * SIZE)]   # 256 results, each 0 or 1
grid = np.array(pixels).reshape(SIZE, SIZE)            # fold the list into a square
show_art(grid, colors=["black", "white"], title="Quantum Noise")

In [ ]:
check_mission_1(qc)

## Mission 2: Four colors from two qubits

Two qubits give four results, `00` to `11`, which become the numbers 0 to 3, so each pixel can be one of four colors.

**TODO:** Put both qubits in superposition.

In [ ]:
qc = QuantumCircuit(2)
# TODO: put BOTH qubits in superposition

qc.measure_all()

pixels = [int(b, 2) for b in run_many(qc, SIZE * SIZE)]   # '00'->0, '01'->1, '10'->2, '11'->3
grid = np.array(pixels).reshape(SIZE, SIZE)
show_art(grid, colors=["#0b132b", "#1c7293", "#f4a259", "#f25c54"], title="Four-Color Quantum Quilt")

In [ ]:
check_mission_2(qc)

## Mission 3: Night sky

Stars should be rare. A tilt with `qc.ry(theta, 0)` sets the chance of measuring 1:

$$P(1) = \sin^2(\theta/2)$$

**TODO:** Find `theta` so only 5% of pixels are stars. Hint: you need $\sin(\theta/2) = \sqrt{0.05}$, and `math.asin` undoes `sin`.

In [ ]:
SKY = 32
star_chance = 0.05

theta = math.pi / 2     # TODO: this gives 50% stars, way too many!

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

pixels = [int(b) for b in run_many(qc, SKY * SKY)]
grid = np.array(pixels).reshape(SKY, SKY)
show_art(grid, colors=["#0a0a23", "#fff5b5"], title="Quantum Night Sky")
print("Chance of a star:", probabilities(qc).get("1", 0))

In [ ]:
check_mission_3(qc)

## Mission 4: Entangled mirror art

Entangled qubits always agree. Use qubit 0 to paint a pixel on the left half and qubit 1 to paint the matching pixel on the right half, flipped like a mirror.

Without entanglement, the two halves are unrelated noise. With entanglement, you get perfect symmetry, like an inkblot test, even though every single pixel is random.

**TODO:** Make `entangled` a Bell pair.

In [ ]:
HALF = 8   # each half is 16 tall and 8 wide

def mirror_art(qc, title, colors=["#222222", "#ff6f91"]):
    results = run_many(qc, SIZE * HALF)                                  # each result looks like '01'
    left = np.array([int(r[1]) for r in results]).reshape(SIZE, HALF)    # qubit 0 = right digit
    right = np.array([int(r[0]) for r in results]).reshape(SIZE, HALF)   # qubit 1 = left digit
    grid = np.hstack([left, np.fliplr(right)])                           # mirror the right half
    show_art(grid, colors=colors, title=title)

# Two independent quantum coins
plain = QuantumCircuit(2)
plain.h([0, 1])
plain.measure_all()
mirror_art(plain, "Not Entangled")

# An entangled pair
entangled = QuantumCircuit(2)
# TODO: make a Bell pair (H on qubit 0, then CNOT from qubit 0 to qubit 1)

entangled.measure_all()
mirror_art(entangled, "Entangled")

In [ ]:
check_mission_4(entangled)

## Mission 5: Quantum gradient

Give every column its own tilt: the first column uses `theta = 0` (always 0), the last uses `theta = math.pi` (always 1), and the columns in between change smoothly. The result fades from one color to the other, with quantum speckle.

**TODO:** Set `theta` so it grows evenly from 0 in the first column to `math.pi` in the last.

In [ ]:
WIDTH, HEIGHT = 32, 16
columns = []

for col in range(WIDTH):
    # TODO: theta goes from 0 (col = 0) up to math.pi (col = WIDTH - 1)
    theta = math.pi / 2
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.measure_all()
    columns.append([int(b) for b in run_many(qc, HEIGHT)])

grid = np.array(columns).T     # turn the list of columns into rows
show_art(grid, colors=["#1d3557", "#f1faee"], title="Quantum Gradient")

In [ ]:
check_mission_5(grid)

## Make it yours: your gallery piece

Create an original piece for the demo. Combine techniques, pick a palette, and give it a **title** and a **three-sentence artist statement** that explains which quantum effect made it. Some directions:

- **Kaleidoscope.** Entangle 4 qubits (all agree, like `0000` or `1111`) and mirror the image in four directions.
- **Gradients in two directions.** Change `theta` by row *and* column, or in a circle from the center.
- **Interference patterns.** Try H, then `ry(theta)`, then H again, with `theta` changing across the canvas. What shapes appear?
- **Quantum skyline.** Use counts from `run()` as building heights in a bar chart.
- **Quantum music.** Map 3-qubit results (0 to 7) to the 8 notes of a scale and write a melody. To hear it, build a sound wave with NumPy and play it with `from IPython.display import Audio`.
- **Series.** Make 3 to 4 related pieces and present them as a mini exhibition.

## Explain it

Answer in your own words and be ready to say them out loud.

1. What sets how many stars appear in your night sky?
2. How does entanglement create the symmetry in your mirror art?
3. Which quantum effect is the star of your gallery piece, and how did you use it?

Our answers: